# Notebook #1.5: Training

## VCPO on Llama Code Demo

In this notebook, we'll:

* Load in the baseline ~8B parameter Llama 3.1 model
* Load in the LIMA dataset
* Add LoRa adapters to the model
* Extract LoRA variance initializations using both techniques:
    1. Laplace Approximation (Primary)
        - Wrap LoRA model with Laplace-Torch Laplace wrapper using hessian_structure of "diag"
        - Fit the model to the Dataset.
        - Call la.optimize_prior_precision()
    2. Adapted MOPED
* Output is the Prior's LoRA variances for:
    1. KL divergence between prior and model for penalty term in loss
    2. Initialize the model.

# Imports, installation, and setup

In [1]:
import datasets


In [2]:
import os
# Force PyTorch to use the exact same numbering as nvidia-smi
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
# Now select GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import urllib3
import httpx
import requests

# 1. Silence the terminal spam of insecure connection warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Workaround to get dataset HTTPX (Hugging Face Hub client)
old_httpx_init = httpx.Client.__init__
def new_httpx_init(self, *args, **kwargs):
    kwargs['verify'] = False # Forcefully disable SSL verification
    old_httpx_init(self, *args, **kwargs)
httpx.Client.__init__ = new_httpx_init

# 3. Workaround to get async HTTPX (Used for parallel downloads)
old_async_init = httpx.AsyncClient.__init__
def new_async_init(self, *args, **kwargs):
    kwargs['verify'] = False
    old_async_init(self, *args, **kwargs)
httpx.AsyncClient.__init__ = new_async_init

# 4. Workaround to get dataset HTTPX (Hugging Face Hub client)
old_request = requests.Session.request
def new_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return old_request(self, method, url, **kwargs)
requests.Session.request = new_request

In [3]:
# this cell should take ~15 seconds

import torch
print(f"Active GPU: {torch.cuda.get_device_name(0)}")

from datetime import datetime
import json
from os import path
import random
import re
import shutil
import typing

import datasets
import huggingface_hub
from datasets import load_dataset, get_dataset_split_names
import peft
from peft import TaskType
import torch
from torch import cuda
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import trl
from laplace import Laplace


Active GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition


In [4]:
import laplace
print(list(laplace.__path__))

['/home/scs/jatho/miniforge3/envs/SI486/lib/python3.12/site-packages/laplace']


# Hugging Face and Dataset Loading

In [5]:
import huggingface_hub
from datasets import load_dataset, get_dataset_split_names


In [6]:

# 1. Login
huggingface_hub.login()


In [7]:

# 2. Define the exact repository
repo_id = "64bits/lima_vicuna_format"

# 3. Ask the Hub what splits exist to avoid guessing
available_splits = get_dataset_split_names(repo_id)
print(f"Available splits found: {available_splits}")

# 4. Load the first split and FORCE a fresh download to bypass corrupted caches
dataset = load_dataset(repo_id, split=available_splits[0], download_mode="force_redownload")

print(f"LIMA Dataset successfully downloaded and loaded! Total prompts: {len(dataset)}")

Available splits found: ['train']


lima_vicuna_format.json:   0%|          | 0.00/3.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1030 [00:00<?, ? examples/s]

LIMA Dataset successfully downloaded and loaded! Total prompts: 1030


In [8]:
if not path.exists('data'):
    os.mkdir("data")

# IO functions

In [9]:
JSONable = typing.Union[dict, list]

def save_jsonable_object_locally(obj: JSONable, filename: str, save_dir: str = "data") -> None:
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, f"{filename}.json")
    print(f"Saving object locally to {filepath}")
    with open(filepath, "w") as f:
        json.dump(obj, f, indent=2)

def load_jsonable_object_locally(filename: str, load_dir: str = "data") -> JSONable:
    filepath = os.path.join(load_dir, f"{filename}.json")
    print(f"Reading {filepath} into Python")
    with open(filepath, "r") as f:
        return json.load(f)

# Loading in the model and applying LoRa weights

In [10]:
import peft
from peft import TaskType
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LENGTH = 256
LORA_RANK = 64
MODEL_NAME = "meta-llama/meta-Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [11]:

# Apply LoRA with PEFT
lora_config = peft.LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    init_lora_weights="olora",
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)
lora_model = peft.get_peft_model(model, lora_config)

Code to see total number of parameters:

```python
# Count up total parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Total params: {total_params:,}")
```

```
Total params: 8,198,033,408
```

Code to see total number of parameters:

```python
# Count up trainable vs. total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params:      {total_params:,}")
print(
    f"Trainable params:  {trainable_params:,} "
    f"({trainable_params/total_params*100:.4f}% of total)"
)
```

```
Total params:      8,198,033,408
Trainable params:  167,772,160 (2.0465% of total)
```

In [12]:
# Use Laplace Approximation to initialize LoRA matrices
# lora_model.eval()
# lora_la_model = Laplace(
#     lora_model,
#     likelihood="classification",
#     subset_of_weights="all",
#     hessian_structure="diag",
# )

In [13]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 1. Translate from LIMA Vicuna format to hugging face format 
#    Tokenize the conversations using the Llama 3.1 template
#    Mask

response_template = "<|start_header_id|>assistant<|end_header_id|>\n\n"
response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

def tokenize_and_mask(examples):
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []
    
    for conv in examples["conversations"]:
        # Map Vicuna format to Hugging Face standard
        standard_conv = [
            {"role": "user" if msg["from"] == "human" else "assistant", "content": msg["value"]}
            for msg in conv
        ]
        
        # Apply Llama 3.1 template and tokenize
        text = tokenizer.apply_chat_template(standard_conv, tokenize=False)
        tokenized = tokenizer(text, truncation=True, max_length=1024)
        
        input_ids = tokenized["input_ids"]
        labels = input_ids.copy()
        
        # MANUALLY MASK THE PROMPT FOR SFT (Replaces the removed TRL collator)
        # Search for the assistant header token sequence
        assistant_start_idx = -1
        window_size = len(response_token_ids)
        
        for i in range(len(input_ids) - window_size + 1):
            if input_ids[i:i+window_size] == response_token_ids:
                assistant_start_idx = i + window_size
                break
        
        # Mask everything before the assistant's actual response with -100
        # (PyTorch natively ignores -100 during loss calculation)
        if assistant_start_idx != -1:
            labels[:assistant_start_idx] = [-100] * assistant_start_idx
        
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(tokenized["attention_mask"])
        batch_labels.append(labels)
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels
    }

print("Tokenizing and Masking LIMA dataset...")
tokenized_dataset = dataset.map(
    tokenize_and_mask, 
    batched=True, 
    remove_columns=dataset.column_names 
)


# We only want the parameter curvature for the next token of the assistant's response, not the prompt
# Use the standard Seq2Seq collator to handle padding (it pads labels with -100 automatically)
collate_fn = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)


# 3. Basic Dataloader

# Batch size 1 is required to prevent the exact Fisher Information matrix from blowing up VRAM
dataloader = DataLoader(
    tokenized_dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=collate_fn
)

Tokenizing and Masking LIMA dataset...


Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

In [14]:
from laplace import Laplace
from laplace.curvature import AsdlEF # recommended in laplace-torch website
from torch import nn

# Wrapper to unpack dictionary and shift the logits
class LaplaceHuggingFaceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, batch_dict):
        # Laplace natively passes the whole MutableMapping (dict) if the model accepts it
        logits = self.model(
            input_ids=batch_dict['input_ids'],
            attention_mask=batch_dict['attention_mask']
        ).logits
        
        # Shift logits to predict the *next* token
        shift_logits = logits[..., :-1, :].contiguous()
        
        # Flatten to 2D for Laplace's native classification backend
        return shift_logits.view(-1, shift_logits.size(-1))

class LaplaceDataLoaderWrapper:
    def __init__(self, dataloader):
        self.dataloader = dataloader
        self.dataset = dataloader.dataset
        
    def __iter__(self):
        for batch in self.dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Shift the labels to match the shifted logits above
            shift_labels = batch['labels'][..., 1:].contiguous()
            batch['shifted_labels'] = shift_labels.view(-1)
            
            yield batch
            
    def __len__(self):
        return len(self.dataloader)

In [20]:
# If we want MOPED, 
# Use MOPED's get rho to determine good variance initialization in LoRA matrices
# def get_rho(sigma, delta):
#     """
#     sigma is represented by softplus function  'sigma = log(1 + exp(rho))' to make sure it 
#     remains always positive and non-transformed 'rho' gets updated during backprop.
#     """
#     rho = torch.log(torch.expm1(delta * torch.abs(sigma)) + 1e-20)
#     return rho

In [ ]:
wrapped_model = LaplaceHuggingFaceWrapper(lora_model).to(device)
laplace_loader = LaplaceDataLoaderWrapper(dataloader)

# 2. Use the native Hugging Face dictionary integration
lora_la_model = Laplace(
    wrapped_model,
    likelihood="classification",
    subset_of_weights="all",     # Laplace natively ignores frozen base weights and isolates LoRA
    hessian_structure="diag",
    dict_key_x="input_ids",      # Tells Laplace the input is a dict and where to trace it
    dict_key_y="shifted_labels", # Tells Laplace where to find the aligned targets
    backend=AsdlEF               # ASDL Empirical Fisher backend prevents the PEFT OOM
)

print("Calculating the Hessian curvature of the LoRA parameters...")
lora_la_model.fit(laplace_loader)
print("Curvature extraction complete.")
lora_la_model.optimize_prior_precision()
print("Prior precision optimization complete.")


Calculating the Hessian curvature of the LoRA parameters...


/home/scs/jatho/miniforge3/envs/SI486/lib/python3.12/site-packages/torch/nn/modules/module.py:1869: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Curvature extraction complete.


/home/scs/jatho/miniforge3/envs/SI486/lib/python3.12/site-packages/laplace/baselaplace.py:435: UserWarning: By default `link_approx` is `probit`. Make sure to set it equals to the way you want to call `la(test_data, pred_type=..., link_approx=...)`.
  warnings.warn(


Prior precision optimization complete.


In [21]:
print("Extracting and structuring Bayesian prior values...")

# 1. Grab the flat 1D tensors from Laplace
flat_means = lora_la_model.mean.detach().cpu()
flat_variances = lora_la_model.posterior_variance.detach().cpu()

structured_prior_dict = {}
current_idx = 0

# 2. Iterate through the exact parameters Laplace evaluated (the trainable ones)
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        num_elements = param.numel()
        param_shape = param.shape
        
        # 3. Slice the flat 1D tensors
        sliced_mean = flat_means[current_idx : current_idx + num_elements]
        sliced_var = flat_variances[current_idx : current_idx + num_elements]
        
        # 4. Reshape the slice back into the 2D matrix geometry (e.g., [64, 4096])
        structured_prior_dict[name] = {
            "mean": sliced_mean.view(param_shape),
            "variance": sliced_var.view(param_shape)
        }
        
        # Move the cursor forward
        current_idx += num_elements

# Quick safety check to ensure we sliced exactly the right amount
assert current_idx == flat_means.numel(), "Mismatch in parameter counts during slicing!"

# 5. Save the geometrically structured dictionary to disk
torch.save(structured_prior_dict, "data/structured_prior_dict_olora.pt")
print(f"Task 2.2 Complete! Structured prior saved with {len(structured_prior_dict)} active matrices.")

Extracting and structuring Bayesian prior values...
Task 2.2 Complete! Structured prior saved with 448 active matrices.


In [ ]:
# # HARD MEMORY CLEAN UP
# del lora_la_model, wrapped_model, laplace_loader
# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [22]:
# Olora structured prior dictionary
import pprint
pprint.pprint(structured_prior_dict)

{'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight': {'mean': tensor([[-7.6953e-01, -1.7212e-02,  1.4465e-02,  ...,  1.1230e-02,
          1.6098e-03, -1.5076e-02],
        [ 0.0000e+00,  7.6953e-01,  9.5215e-03,  ..., -1.3000e-02,
         -2.6489e-02,  2.2888e-03],
        [ 0.0000e+00,  0.0000e+00, -7.3828e-01,  ..., -7.3624e-04,
          1.5747e-02, -4.4250e-03],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -2.3804e-03,
         -1.2360e-03,  1.5991e-02],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -2.8381e-03,
         -5.0964e-03,  1.1536e-02],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0437e-02,
         -5.2795e-03, -9.9487e-03]]),
                                                                         'variance': tensor([[0.0047, 0.0047, 0.0046,  ..., 0.0046, 0.0046, 0.0047],
        [0.0047, 0.0047, 0.0047,  ..., 0.0046, 0.0047, 0.0047],
        [0.0046, 0.0047, 0.0046,  ..., 0.0046, 0.0046, 0.0047],
        .

In [ ]:
# # pissa 
# import pprint
# pprint.pprint(structured_prior_dict)

{'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight': {'mean': tensor([[-0.0096, -0.0115,  0.0056,  ...,  0.0109, -0.0053, -0.0017],
        [-0.0023, -0.0006,  0.0178,  ..., -0.0066,  0.0013,  0.0017],
        [ 0.0193,  0.0139, -0.0239,  ...,  0.0073,  0.0227,  0.0103],
        ...,
        [ 0.0063, -0.0002, -0.0124,  ...,  0.0049,  0.0054, -0.0013],
        [ 0.0094, -0.0210,  0.0183,  ..., -0.0153, -0.0101,  0.0067],
        [-0.0062, -0.0204,  0.0079,  ...,  0.0442,  0.0103,  0.0309]]),
                                                                         'variance': tensor([[0.0049, 0.0049, 0.0047,  ..., 0.0048, 0.0049, 0.0050],
        [0.0041, 0.0038, 0.0040,  ..., 0.0036, 0.0017, 0.0042],
        [0.0040, 0.0043, 0.0041,  ..., 0.0036, 0.0020, 0.0044],
        ...,
        [0.0048, 0.0049, 0.0049,  ..., 0.0044, 0.0043, 0.0049],
        [0.0047, 0.0049, 0.0047,  ..., 0.0044, 0.0043, 0.0048],
        [0.0048, 0.0048, 0.0046,  ..., 0.0042, 0.0041, 0.0048]],
 

In [23]:
from reparam_LoRA_Skeleton import BayesianLoraLinear
# Build Reparameterization Model Calling 

# 1. Execute PEFT code that replaces the linear layers in the base model with the updated weights compensating for OLORA.
# Loop through the model's modules and replace the linear layers with the reparameterized ones

# Apply LoRA with PEFT
bayes_config = peft.LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    init_lora_weights="olora",
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)
# Set config.custommapping to the structured prior dictionary
custom_module_mapping = {nn.Linear: BayesianLoraLinear}
# register the new mapping
bayes_config._register_custom_module(custom_module_mapping)

bayes_model = peft.get_peft_model(model, bayes_config)


/home/scs/jatho/miniforge3/envs/SI486/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/scs/jatho/miniforge3/envs/SI486/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# import os
# from peft import PeftModel

# # Assume 'model' is your trained PeftModel instance
# output_dir = "./my_total_model"

# # 1. Save the base model weights separately
# # This extracts the original underlying model and writes its full gigabyte-sized files
# model.get_base_model().save_pretrained(os.path.join(output_dir, "base_model"))

# # 2. Save the adapter weights into their own subfolder
# # This writes the lightweight adapter_config.json and adapter_model.safetensors
# model.save_pretrained(os.path.join(output_dir, "adapters"))


In [17]:
# Save Model Adapter
torch.save(bayes_model, "data/bayes_model_olora_whole.pt")


In [17]:
def _is_valid_number(candidate: str) -> bool:
    """
    Check if 'candidate' matches one of the patterns:
    - plain integer (e.g. '39')
    - dollar + integer (e.g. '$39')
    - comma-separated integer (e.g. '43,500')
    """
    candidate = candidate.strip()
    # e.g. remove leading/trailing '$', but handle the case if it starts with $
    # We'll use a regex approach to handle commas or $ sign:
    pattern = r"^\$?\d{1,3}(,\d{3})*(\.\d+)?$"
    return bool(re.match(pattern, candidate))


# from 
# https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb
def _extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()


def _extract_hash_answer(text: str) -> str:
    return text.split("####")[1].strip()


def _random_three_alphanumeric():
    # 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
    chars = string.ascii_letters + string.digits
    return ''.join(random.choices(chars, k=3))


def generate_answers(
    model,
    tokenizer,
    system_prompt: str,
    dataset_subset,
    num_attempts: int = 5,
    max_length_prompt: int = 256,
    max_new_tokens: int = 256,
    start_index: int = 0,
    end_index: typing.Optional[int] = None,
):
    """
    Generate multiple answers for each sample in a dataset subset using a specified model,
    and upload the results to S3.

    Args:
        model: The model to use for generation (e.g., baseline model or fine-tuned LoRA model).
        tokenizer: The tokenizer corresponding to the model.
        system_prompt: The system prompt to prepend to each question.
        dataset_subset: The dataset subset to process (e.g., 100 shuffled GSM8K test samples).
        local_prefix: Prefix for local file paths (e.g., 'gsm8k_llama_7b_100_record_test_').
        num_attempts: Number of answer attempts per sample (default: 5).
        max_length_prompt: Maximum token length for the input prompt (default: 256).
        max_new_tokens: Maximum new tokens to generate (default: 256).
        start_index: Index to start processing from (default: 1, useful for resuming).
    """
    # Set model to evaluation mode
    model.eval()

    # Move model to CUDA
    model.to("cuda")

    # Handle index bounds
    dataset_len = len(dataset_subset)
    start_index = max(1, start_index)
    end_index = dataset_len if end_index is None else min(dataset_len, end_index)

    # Process each sample in the specified range
    for i in range(start_index, end_index):
        sample = dataset_subset[i - 1]

        # Extract question and gold answer from the sample
        question_text = sample["question"]
        gold_raw = sample["answer"]
        gold_answer = _extract_hash_answer(gold_raw)

        # Construct the prompt based on use_chat_template
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question_text},
        ]
        full_prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Tokenize the prompt
        inputs = tokenizer(
            full_prompt_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length_prompt,
        ).to("cuda")

        # Get the length of the prompt tokens
        prompt_len = inputs["input_ids"].shape[1]

        # Generate `num_attempts` answers for the sample
        sampled_answers = []
        for _ in range(num_attempts):
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.95,
                )

            # Extract newly generated tokens (after the prompt)
            gen_tokens = outputs[0, prompt_len:]
            completion_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

            # Extract and validate the predicted answer
            predicted_answer = _extract_xml_answer(completion_text)
            if _is_valid_number(predicted_answer):
                pred_clean = predicted_answer.replace("$", "").replace(",", "").strip()
                gold_clean = gold_answer.replace("$", "").replace(",", "").strip()
                correct = pred_clean == gold_clean
            else:
                correct = False

            # Store attempt details
            sampled_answers.append(
                {
                    "raw_text": completion_text,
                    "predicted_answer": predicted_answer,
                    "valid_number": _is_valid_number(predicted_answer),
                    "correct": correct,
                }
            )

        # Compile the record for this sample
        record = {
            "index": i,
            "question": question_text,
            "gold_answer": gold_answer,
            "model_outputs": sampled_answers,
        }

        now = datetime.now()

        timestamp_str = now.strftime("%b-%d-%Y_%I-%M-%S%p")

        filename = f"{_random_three_alphanumeric()}_{timestamp_str}"
        # Save to local drive
        save_jsonable_object_locally(
            record, filename=filename, save_dir="data/3-FineTune-Results"
        )


In [24]:
import string
gsm8k_data = datasets.load_dataset('openai/gsm8k', 'main')
test_set = gsm8k_data['test']
test_set_shuffled = test_set.shuffle(seed=250217)

SYSTEM_PROMPT = """
### EXAMPLE ###
Q: 3+2
<reasoning>
3 plus 2 is 5
</reasoning>
<answer>
5
</answer>

Now follow the same format EXACTLY for each question:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

local_prefix = 'gsm8k_lora_VCPO_prefinetrain'

generate_answers(
    model=bayes_model,
    tokenizer=tokenizer,
    system_prompt=SYSTEM_PROMPT,
    dataset_subset=test_set_shuffled,
    start_index=1,
    end_index=5,
)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saving object locally to data/3-FineTune-Results/CAj_Jul-15-2026_06-16-41PM.json


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

In [22]:
print(bayes_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:

# 2. Replace the KL divergence loss, such that it takes the KL between the prior dictionary of LoRA distributions and the present model's LoRA distributions.



# 3. Ready to Train!

# If desired, try with adapted MOPED algorithm to initialize the LoRA matrices with a good variance.
# - Remember to go back and initialize GRPO base model with OLORA for a good baseline.

# Set means of the prior LoRA layers to "0"

****************************** End Notebook *******************************